In [2]:
!ls -l /root/merged_spinkick/merge_all_spinkick_mugenmixamo/train/v_0a6012c77337a2fb63dc8e0b7f34728bcd76bd01e62ec6ad6461d7a0ce16ef47_dir/


total 212
-rw-r--r-- 1 root root 8413 Apr  5  2024 v_0a6012c77337a2fb63dc8e0b7f34728bcd76bd01e62ec6ad6461d7a0ce16ef47_0000.jpg
-rw-r--r-- 1 root root  191 Apr  5  2024 v_0a6012c77337a2fb63dc8e0b7f34728bcd76bd01e62ec6ad6461d7a0ce16ef47_0000.txt
-rw-r--r-- 1 root root 8413 Apr  5  2024 v_0a6012c77337a2fb63dc8e0b7f34728bcd76bd01e62ec6ad6461d7a0ce16ef47_0001.jpg
-rw-r--r-- 1 root root  191 Apr  5  2024 v_0a6012c77337a2fb63dc8e0b7f34728bcd76bd01e62ec6ad6461d7a0ce16ef47_0001.txt
-rw-r--r-- 1 root root 7588 Apr  5  2024 v_0a6012c77337a2fb63dc8e0b7f34728bcd76bd01e62ec6ad6461d7a0ce16ef47_0002.jpg
-rw-r--r-- 1 root root  191 Apr  5  2024 v_0a6012c77337a2fb63dc8e0b7f34728bcd76bd01e62ec6ad6461d7a0ce16ef47_0002.txt
-rw-r--r-- 1 root root 7588 Apr  5  2024 v_0a6012c77337a2fb63dc8e0b7f34728bcd76bd01e62ec6ad6461d7a0ce16ef47_0003.jpg
-rw-r--r-- 1 root root  191 Apr  5  2024 v_0a6012c77337a2fb63dc8e0b7f34728bcd76bd01e62ec6ad6461d7a0ce16ef47_0003.txt
-rw-r--r-- 1 root root 7263 Apr  5  2024 v_0a6012c7733

In [3]:
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 27.0 MB/s eta 0:00:00:00:0100:01


In [12]:
import os
import re
import cv2
from collections import defaultdict

def process_directory(directory_path):
    """
    Process a directory to merge image files into MP4 videos and create corresponding text files.
    
    Args:
        directory_path (str): Path to the directory containing image and text files
    """
    # Get all files in the directory
    files = os.listdir(directory_path)
    
    # Group files by base name (everything before the sequence number)
    image_groups = defaultdict(list)
    base_names = set()
    
    for file in files:
        # Extract base name and sequence number
        match = re.match(r'(.+)_(\d+)\.(jpg|txt)$', file)
        if match:
            base_name, seq_num, file_type = match.groups()
            base_names.add(base_name)
            
            if file_type == 'jpg':
                image_groups[base_name].append((int(seq_num), os.path.join(directory_path, file)))
    
    # Process each group of image files
    for base_name in base_names:
        # Sort image files by sequence number
        sorted_images = sorted(image_groups[base_name], key=lambda x: x[0])
        
        if sorted_images:
            # Get paths to all image files
            image_paths = [file_path for _, file_path in sorted_images]
            
            # Create video from images
            output_video_path = os.path.join(directory_path, f"{base_name}.mp4")
            create_video_from_images(image_paths, output_video_path)
            
            # Create text file using content from _0000.txt
            source_txt_path = os.path.join(directory_path, f"{base_name}_0000.txt")
            target_txt_path = os.path.join(directory_path, f"{base_name}.txt")
            
            if os.path.exists(source_txt_path):
                with open(source_txt_path, 'r') as src_file:
                    content = src_file.read()
                
                with open(target_txt_path, 'w') as target_file:
                    target_file.write(content)
                print(f"Created text file: {target_txt_path}")
            else:
                print(f"Warning: Source text file {source_txt_path} not found")

def create_video_from_images(image_paths, output_path, fps=1):
    """
    Create a video from a list of image paths, with each image as exactly one frame.
    
    Args:
        image_paths (list): List of paths to the image files
        output_path (str): Path where the output video will be saved
        fps (int): Frames per second for the output video (default: 1 fps for better viewing)
    """
    if not image_paths:
        print("No images found to create video")
        return
    
    # Read the first image to get dimensions
    img = cv2.imread(image_paths[0])
    height, width, layers = img.shape
    
    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Write each image to the video - one frame per image
    frame_count = 0
    for image_path in image_paths:
        img = cv2.imread(image_path)
        if img is not None:
            video.write(img)
            frame_count += 1
        else:
            print(f"Warning: Could not read image {image_path}")
    
    # Release the video writer
    video.release()
    print(f"Video created: {output_path} with {frame_count} frames from {len(image_paths)} images")
    
if False and __name__ == "__main__":
    import sys
    
    if len(sys.argv) != 2:
        print("Usage: python process_directory.py <directory_path>")
        sys.exit(1)
    
    directory_path = sys.argv[1]
    if not os.path.isdir(directory_path):
        print(f"Error: {directory_path} is not a directory")
        sys.exit(1)
    
    process_directory(directory_path)

In [3]:
import os
import sys
import argparse
#from process_directory import process_directory

def process_all_subdirectories(root_path):
    """
    Process all subdirectories under the given root path.
    
    Args:
        root_path (str): Path to the root directory
    """
    # Check if the root path exists
    if not os.path.isdir(root_path):
        print(f"Error: {root_path} is not a directory")
        return False
    
    # Get all subdirectories
    subdirectories = [os.path.join(root_path, d) for d in os.listdir(root_path) 
                     if os.path.isdir(os.path.join(root_path, d))]
    
    # If no subdirectories, process the root directory itself
    if not subdirectories:
        print(f"No subdirectories found in {root_path}, processing root directory.")
        process_directory(root_path)
        return True
    
    # Process each subdirectory
    print(f"Found {len(subdirectories)} subdirectories to process.")
    for subdir in subdirectories:
        print(f"Processing directory: {subdir}")
        process_directory(subdir)
    
    return True

if False and __name__ == "__main__":
    parser = argparse.ArgumentParser(description='Process image and text files in directories.')
    parser.add_argument('root_path', help='Root directory path containing subdirectories to process')
    
    args = parser.parse_args()
    
    if process_all_subdirectories(args.root_path):
        print("All directories processed successfully.")
    else:
        sys.exit(1)

In [13]:
process_all_subdirectories('/root/merged_spinkick/merge_all_spinkick_mugenmixamo/train')

Found 421 subdirectories to process.
Processing directory: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/train/v_00df914571acdb2a725121f4f4b702f9389a0d043be0381448aa08a0e71b8079_dir
Video created: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/train/v_00df914571acdb2a725121f4f4b702f9389a0d043be0381448aa08a0e71b8079_dir/v_00df914571acdb2a725121f4f4b702f9389a0d043be0381448aa08a0e71b8079.mp4 with 16 frames from 16 images
Created text file: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/train/v_00df914571acdb2a725121f4f4b702f9389a0d043be0381448aa08a0e71b8079_dir/v_00df914571acdb2a725121f4f4b702f9389a0d043be0381448aa08a0e71b8079.txt
Processing directory: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/train/v_01014b58c6fdf5e6167fc63641cdafb3dac70783a1a98ca994699c35bd02a70d_dir
Video created: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/train/v_01014b58c6fdf5e6167fc63641cdafb3dac70783a1a98ca994699c35bd02a70d_dir/v_01014b58c6fdf5e6167fc63641cdafb3dac70783a1a98c

True

In [14]:
process_all_subdirectories('/root/merged_spinkick/merge_all_spinkick_mugenmixamo/val')

Found 19 subdirectories to process.
Processing directory: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/val/v_045d6d2188c1f80d28d23eaabab9aa12fc5a19156351acb15477b228367d2fb0_dir
Video created: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/val/v_045d6d2188c1f80d28d23eaabab9aa12fc5a19156351acb15477b228367d2fb0_dir/v_045d6d2188c1f80d28d23eaabab9aa12fc5a19156351acb15477b228367d2fb0.mp4 with 16 frames from 16 images
Created text file: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/val/v_045d6d2188c1f80d28d23eaabab9aa12fc5a19156351acb15477b228367d2fb0_dir/v_045d6d2188c1f80d28d23eaabab9aa12fc5a19156351acb15477b228367d2fb0.txt
Processing directory: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/val/v_14b9c079b1ee83a27eccfcdf496f954246b6d4d485a0a0e97a10892e0aa88330_dir
Video created: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/val/v_14b9c079b1ee83a27eccfcdf496f954246b6d4d485a0a0e97a10892e0aa88330_dir/v_14b9c079b1ee83a27eccfcdf496f954246b6d4d485a0a0e97a10892e0

True

In [7]:
output_path = '/root/godmodai_hunyuan_dataset/spinkick'

In [8]:
import os
import sys
import shutil
import argparse
from pathlib import Path

def copy_generated_files(root_path, output_path):
    """
    Copy all generated .mp4 files and their corresponding .txt files
    from subdirectories under root_path to the output_path.
    
    Args:
        root_path (str): Path to the root directory containing subdirectories
        output_path (str): Path to the output directory where files will be copied
    
    Returns:
        tuple: Count of (mp4_files_copied, txt_files_copied)
    """
    # Create output directory if it doesn't exist
    if not os.path.exists(output_path):
        os.makedirs(output_path)
        print(f"Created output directory: {output_path}")
    
    # Check if the root path exists
    if not os.path.isdir(root_path):
        print(f"Error: {root_path} is not a directory")
        return 0, 0
    
    mp4_count = 0
    txt_count = 0
    
    # Walk through all subdirectories
    for dirpath, dirnames, filenames in os.walk(root_path):
        # Find all .mp4 files
        mp4_files = [f for f in filenames if f.endswith('.mp4')]
        
        for mp4_file in mp4_files:
            mp4_path = os.path.join(dirpath, mp4_file)
            
            # Find corresponding .txt file (same name but .txt extension)
            txt_file = os.path.splitext(mp4_file)[0] + '.txt'
            txt_path = os.path.join(dirpath, txt_file)
            
            # Copy the .mp4 file
            try:
                shutil.copy2(mp4_path, output_path)
                mp4_count += 1
                print(f"Copied MP4: {mp4_path} -> {output_path}")
            except Exception as e:
                print(f"Error copying {mp4_path}: {e}")
            
            # Copy the .txt file if it exists
            if os.path.exists(txt_path):
                try:
                    shutil.copy2(txt_path, output_path)
                    txt_count += 1
                    print(f"Copied TXT: {txt_path} -> {output_path}")
                except Exception as e:
                    print(f"Error copying {txt_path}: {e}")
            else:
                print(f"Warning: Corresponding text file not found for {mp4_path}")
    
    return mp4_count, txt_count

if False and __name__ == "__main__":
    parser = argparse.ArgumentParser(description='Copy generated MP4 and TXT files to output directory.')
    parser.add_argument('root_path', help='Root directory path containing files to copy')
    parser.add_argument('output_path', help='Output directory path where files will be copied')
    
    args = parser.parse_args()
    
    mp4_copied, txt_copied = copy_generated_files(args.root_path, args.output_path)
    
    print(f"\nSummary:")
    print(f"MP4 files copied: {mp4_copied}")
    print(f"TXT files copied: {txt_copied}")
    
    if mp4_copied > 0:
        print("Files copied successfully.")
    else:
        print("No files were copied.")
        sys.exit(1)

In [15]:
copy_generated_files('/root/merged_spinkick/merge_all_spinkick_mugenmixamo/train', output_path)

Copied MP4: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/train/v_00df914571acdb2a725121f4f4b702f9389a0d043be0381448aa08a0e71b8079_dir/v_00df914571acdb2a725121f4f4b702f9389a0d043be0381448aa08a0e71b8079.mp4 -> /root/godmodai_hunyuan_dataset/spinkick
Copied TXT: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/train/v_00df914571acdb2a725121f4f4b702f9389a0d043be0381448aa08a0e71b8079_dir/v_00df914571acdb2a725121f4f4b702f9389a0d043be0381448aa08a0e71b8079.txt -> /root/godmodai_hunyuan_dataset/spinkick
Copied MP4: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/train/v_01014b58c6fdf5e6167fc63641cdafb3dac70783a1a98ca994699c35bd02a70d_dir/v_01014b58c6fdf5e6167fc63641cdafb3dac70783a1a98ca994699c35bd02a70d.mp4 -> /root/godmodai_hunyuan_dataset/spinkick
Copied TXT: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/train/v_01014b58c6fdf5e6167fc63641cdafb3dac70783a1a98ca994699c35bd02a70d_dir/v_01014b58c6fdf5e6167fc63641cdafb3dac70783a1a98ca994699c35bd02a70d.txt -> /root/godmodai_h

(421, 421)

In [16]:
copy_generated_files('/root/merged_spinkick/merge_all_spinkick_mugenmixamo/val', output_path)

Copied MP4: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/val/v_045d6d2188c1f80d28d23eaabab9aa12fc5a19156351acb15477b228367d2fb0_dir/v_045d6d2188c1f80d28d23eaabab9aa12fc5a19156351acb15477b228367d2fb0.mp4 -> /root/godmodai_hunyuan_dataset/spinkick
Copied TXT: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/val/v_045d6d2188c1f80d28d23eaabab9aa12fc5a19156351acb15477b228367d2fb0_dir/v_045d6d2188c1f80d28d23eaabab9aa12fc5a19156351acb15477b228367d2fb0.txt -> /root/godmodai_hunyuan_dataset/spinkick
Copied MP4: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/val/v_14b9c079b1ee83a27eccfcdf496f954246b6d4d485a0a0e97a10892e0aa88330_dir/v_14b9c079b1ee83a27eccfcdf496f954246b6d4d485a0a0e97a10892e0aa88330.mp4 -> /root/godmodai_hunyuan_dataset/spinkick
Copied TXT: /root/merged_spinkick/merge_all_spinkick_mugenmixamo/val/v_14b9c079b1ee83a27eccfcdf496f954246b6d4d485a0a0e97a10892e0aa88330_dir/v_14b9c079b1ee83a27eccfcdf496f954246b6d4d485a0a0e97a10892e0aa88330.txt -> /root/godmodai_hunyuan_d

(19, 19)